# Beyond: the dependence band shown with every aggregate, and an honest replication verdict

- **Level:** 4, Beyond the state of the art
- **Prerequisites:** notebooks 01 to 03 and 08 of this topic (the ladder, the band, the bounds and the state of the art)
- **Reading time:** about 45 minutes

## Objectives

By the end of this notebook you can:

- state the claim BS-9 of the product precisely (every aggregate capital figure of C18 is shown with the dependence band, never as one diversified number) and what its evidence is;
- check the claim on every committed C18 document that carries an aggregate: eight band rows in a fixed order, one row labelled headline capital and the others information, the closed-form rows recomputed with the engine, the bracket and sign rules, and the Euler allocation adding up;
- see the engine return the **whole band** on a golden example, rather than one number;
- say what a single number would hide, measured on the committed band;
- read the readiness record of the case and say honestly which case of the topic is planned and not built, and why.

## Setup

Install the pinned notebook environment once, in a virtual environment, with the familiar command:

```bash
pip install -r notebooks/requirements.txt
```

or install only the packages this notebook uses, with the same exact pins:

```bash
pip install riskvalidation==0.9.0 numpy==2.4.6 scipy==1.18.1
```

`scripts/setup.sh --notebooks` (`scripts/setup.ps1 -Notebooks` on Windows) builds the whole environment, `.venv-notebooks`, for you. A notebook never installs anything by itself. The engine's test identifiers are printed as the engine writes them. Its messages and notes are written in English: this tree prints them as written, and the Spanish tree prints a translation from a table in the notebook that fails if the engine changes the text.

The next cell imports what the notebook uses and prints the versions of the packages, so that you can compare them with the pins.

In [1]:
# Import the engine and the helpers, and print the versions of the packages this notebook uses
import hashlib
import importlib.metadata as md
import json
from pathlib import Path

import numpy as np
import riskvalidation as rv
from riskvalidation.aggregation import band as rv_band
from riskvalidation.aggregation import copulas as cop
from riskvalidation.aggregation import simple
from riskvalidation.validation import aggregation as agg_tests
from scipy import stats
for name in ("riskvalidation", "numpy", "scipy",):
    print(name, md.version(name))

riskvalidation 0.9.0
numpy 2.4.6
scipy 1.18.1


## Data

This notebook reads every aggregate that **C18** commits, and the records that say what is and is not built.

- **Owner and licence:** the documents are written by this repository's offline pipeline (`data-pipeline/pipeline/cases/c18/`) with the engine `riskvalidation` and committed under `data/derived/C18/`. The scenario paths come from the Federal Reserve (public domain), the EBA and ESRB (reproduction authorised with the source acknowledged) and the NGFS (derived-only: only the few variables the case uses). **Banco Andino is a synthetic bank**: its book and capital are calibrated to public aggregates, and the scenario paths are real United States and euro-area paths transposed onto its jump-off, so the satellites are a US-calibrated adaptation.
- **Truth status:** `synthetic-calibrated`. There is no answer key for the bank; the checks of this notebook are recomputations of committed numbers with the engine, not comparisons with a published result.
- **Baked by:** `riskvalidation` 0.8.1 (seed 42), the version this notebook installs, so a recomputation is expected to match to rounding.
- **Files read:** the four scenario documents `data/derived/C18/baseline.json`, `eba-2025-adverse.json`, `fed-2026-severely-adverse.json` and `ngfs-delayed-transition.json` and the aggregation document `t-copula-baseline.json` (each carries `outputs.aggregation`, and the last also the market-credit stress), the case index `data/derived/manifests/index.json`, the readiness record `data/derived/readiness.json`, and the engine's stress contract `docs/contracts/engine/stress-api.md` (its scope paragraph records why case C19 stays planned).
- **What is claimed and what is not.** The claim is about what **Contraste does**: it shows the band. It is not a claim that nobody else does, which the research could not establish beyond its own search.

In [2]:
# The notebook sits three folders below the repository root; a missing file stops the notebook and names its path
ROOT = Path("../../..").resolve()
DERIVED = ROOT / "data" / "derived"


def read_json(relative):
    return json.loads((DERIVED / relative).read_text(encoding="utf-8"))


print("derived folder found:", DERIVED.is_dir())

SCENARIOS = ("baseline", "eba-2025-adverse", "fed-2026-severely-adverse", "ngfs-delayed-transition")
blocks = {name: read_json(f"C18/{name}.json")["outputs"]["aggregation"] for name in SCENARIOS}
second = read_json("C18/t-copula-baseline.json")["outputs"]
blocks["t-copula-baseline"] = second["aggregation"]
blocks["t-copula-baseline, market-credit 0.8"] = second["correlation_stress"]["aggregation"]
print("documents with an aggregate:", len(blocks))
for name in blocks:
    print(" -", name)

derived folder found: True
documents with an aggregate: 6
 - baseline
 - eba-2025-adverse
 - fed-2026-severely-adverse
 - ngfs-delayed-transition
 - t-copula-baseline
 - t-copula-baseline, market-credit 0.8


## Method

### Step 1: the claim, checked on every aggregate

**The claim (BS-9).** Every aggregate capital figure of C18 is shown with the **band** of aggregation methods over the same marginals, one horizon and one confidence level: the lower bound, the simple sum, the square-root rule, the Gaussian copula, one t copula for each degree of freedom of the named set, and the sharp worst case by the rearrangement algorithm, in that order. The simple sum is the only row labelled **headline capital**; every other row, the Euler allocation and the model-risk measures are labelled **information, not capital**. The case never shows one diversified number.

The next cell checks the claim structurally on the five documents (six aggregates, counting the correlation stress, of which only two are distinct, which the cell shows by hashing them): the number and order of the rows, the status of each, that exactly one row is the headline and it is the simple sum, that the Euler contributions add up, and that the model-risk block is labelled information. It prints a line per document.

In [3]:
# Check the structure of the band in every committed aggregate: rows, order, status, headline, Euler additivity, model-risk label
EXPECTED = ["lower-bound", "simple-sum", "variance-covariance", "gaussian-copula", "t-copula", "t-copula", "t-copula", "worst-case"]
for name, block in blocks.items():
    methods = [row["method"] for row in block["band"]]
    headline_rows = [row["method"] for row in block["band"] if row["status"] == "headline-capital"]
    others = {row["status"] for row in block["band"] if row["method"] != "simple-sum"}
    worst = next(row["capital_clp_m"] for row in block["band"] if row["method"] == "worst-case")
    print(f"{name:38s} rows {len(methods)} | order as stated: {methods == EXPECTED} | headline rows: {headline_rows} | the others: {sorted(others)}")
    print(f"{'':38s} headline {block['headline']['capital_clp_m']:,.0f} | worst case over headline {worst / block['headline']['capital_clp_m']:.4f} | Euler gap {block['euler']['additivity']['relative_error']:.1e} | model risk: {block['model_risk']['status']} | one level and horizon: {block['confidence_level']}, {block['horizon_years']}")
# The aggregation does not depend on the scenario: show which of the six blocks are one and the same block
digest = {name: hashlib.sha256(json.dumps(block, sort_keys=True).encode("utf-8")).hexdigest() for name, block in blocks.items()}
stressed_name = "t-copula-baseline, market-credit 0.8"
same_as_baseline = [name for name in blocks if digest[name] == digest["baseline"]]
print("blocks identical to the baseline one:", same_as_baseline)
print("distinct aggregates among the six:", len(set(digest.values())), "| sha256 of the baseline block:", digest["baseline"][:12], "| of the stressed block:", digest[stressed_name][:12])

baseline                               rows 8 | order as stated: True | headline rows: ['simple-sum'] | the others: ['information-not-capital']
                                       headline 658,046 | worst case over headline 1.2512 | Euler gap 0.0e+00 | model risk: information-not-capital | one level and horizon: 0.999, 1
eba-2025-adverse                       rows 8 | order as stated: True | headline rows: ['simple-sum'] | the others: ['information-not-capital']
                                       headline 658,046 | worst case over headline 1.2512 | Euler gap 0.0e+00 | model risk: information-not-capital | one level and horizon: 0.999, 1
fed-2026-severely-adverse              rows 8 | order as stated: True | headline rows: ['simple-sum'] | the others: ['information-not-capital']
                                       headline 658,046 | worst case over headline 1.2512 | Euler gap 0.0e+00 | model risk: information-not-capital | one level and horizon: 0.999, 1
ngfs-delayed-transitio

### Step 2: recomputing what can be recomputed, on every aggregate

Structure is not enough: the rows must be right. Two rows of the band depend only on the marginals' capitals and the correlation matrix, so the engine recomputes them exactly from each document: the simple sum and the square-root rule. The rules that bind the rest are the engine's tests: `worst_case_bracket` (every row inside the bracket) and `diversification_sign` (no diversified row above the simple sum), and the stressed document must have a larger square-root row than the base one, because a higher correlation cannot lower the capital. The next cell runs all of that on the six aggregates.

In [4]:
# Recompute the simple sum and the square-root row of each aggregate and run the bracket and sign tests on it
for name, block in blocks.items():
    ec = np.array([m["ec_clp_m"] for m in block["marginals"]])
    corr = np.array(block["correlation"]["matrix"])
    rows = {(r["method"], r["nu"]): r["capital_clp_m"] for r in block["band"]}
    exact = bool(np.isclose(simple.simple_sum(ec), rows[("simple-sum", None)]) and np.isclose(simple.variance_covariance(ec, corr), rows[("variance-covariance", None)]))
    between = [v for (method, nu), v in rows.items() if method not in ("lower-bound", "worst-case")]
    bracket = agg_tests.worst_case_bracket(rows[("lower-bound", None)], between, rows[("worst-case", None)])
    factors = [r["diversification_factor"] for r in block["band"] if r["method"] in ("variance-covariance", "gaussian-copula", "t-copula")]
    sign = agg_tests.diversification_sign(factors)
    print(f"{name:38s} closed-form rows recomputed exactly: {exact} | bracket: {bracket.light.value} | sign: {sign.light.value}")
base_vc = next(r["capital_clp_m"] for r in blocks["t-copula-baseline"]["band"] if r["method"] == "variance-covariance")
stress_vc = next(r["capital_clp_m"] for r in blocks["t-copula-baseline, market-credit 0.8"]["band"] if r["method"] == "variance-covariance")
print("square-root row, base and stressed:", round(base_vc, 1), round(stress_vc, 1), "| a higher correlation does not lower it:", stress_vc >= base_vc)

baseline                               closed-form rows recomputed exactly: True | bracket: green | sign: green
eba-2025-adverse                       closed-form rows recomputed exactly: True | bracket: green | sign: green
fed-2026-severely-adverse              closed-form rows recomputed exactly: True | bracket: green | sign: green
ngfs-delayed-transition                closed-form rows recomputed exactly: True | bracket: green | sign: green
t-copula-baseline                      closed-form rows recomputed exactly: True | bracket: green | sign: green
t-copula-baseline, market-credit 0.8   closed-form rows recomputed exactly: True | bracket: green | sign: green
square-root row, base and stressed: 528106.4 568520.5 | a higher correlation does not lower it: True


### Step 3: the engine returns the whole band

The claim rests on the engine: `dependence_band` returns a `Band` of rows and a headline, never a scalar, and its docstring gives a golden example: capitals 3 and 4, expected losses 0, correlation 0.5, which must give a simple-sum row of 7.0, a square-root row of 6.0828 and a lower bound at most the square-root row, which is at most the worst case. The next cell builds two normal marginals with those 99.9 percent capitals, runs the band at a reduced size (200,000 draws, 20,000 bound cells), prints the rows with their status, and checks the golden values and the ordering. It also prints the docstring line that records the published example of the worst-case bounds, which notebook 03 reproduced.

In [5]:
# The engine's golden example: two normal marginals with capitals 3 and 4, and the whole band it returns
import inspect
import re

from riskvalidation.aggregation import bounds

z = stats.norm.ppf(0.999)
grid = stats.norm.ppf((np.arange(1, 100_001) - 0.5) / 100_000)
golden = rv_band.dependence_band([cop.marginal_ppf_sample(3 / z * grid), cop.marginal_ppf_sample(4 / z * grid)], np.array([3.0, 4.0]), np.array([0.0, 0.0]),
                                 np.array([[1.0, 0.5], [0.5, 1.0]]), [4.0, 30.0], level=0.999, horizon_years=1, n_draws=200_000, seed=1,
                                 chunk_size=100_000, bounds_n=20_000, bounds_epsilon=1e-3, ec_consistency_tolerance=0.05)
print("returned object:", type(golden).__name__, "| headline:", golden.headline, "| rows:", len(golden.rows))
for row in golden.rows:
    print(f"{row.method:20s} nu {str(row.nu):5s} capital {row.capital:8.4f}  share of the sum {row.diversification_factor:.4f}  {row.status}")
capitals = {(r.method, r.nu): r.capital for r in golden.rows}
print("simple sum 7:", bool(np.isclose(capitals[("simple-sum", None)], 7.0)), "| square-root row 6.0828:", bool(np.isclose(capitals[("variance-covariance", None)], 6.0828, atol=1e-4)),
      "| lower <= square-root <= worst:", capitals[("lower-bound", None)] <= capitals[("variance-covariance", None)] <= capitals[("worst-case", None)])
doc = " ".join(inspect.getdoc(bounds.worst_var_bounds).split())
published = re.search(r"Golden \(Table 4, Pareto\(2\), N = 1e5, epsilon = 1e-3, d = 8\): alpha 0.99 lower ([\d.]+) - ([\d.]+), upper ([\d.]+) - ([\d.]+)", doc).groups()
print("the docstring records the published example (Table 4, Pareto(2), d = 8, alpha 0.99): best VaR", published[0], "to", published[1], "| worst VaR", published[2], "to", published[3])

returned object: Band | headline: 7.0 | rows: 7
lower-bound          nu None  capital   0.7601  share of the sum 0.1086  information-not-capital
simple-sum           nu None  capital   7.0000  share of the sum 1.0000  headline-capital
variance-covariance  nu None  capital   6.0828  share of the sum 0.8690  information-not-capital
gaussian-copula      nu None  capital   6.0912  share of the sum 0.8702  information-not-capital
t-copula             nu 4.0   capital   6.4726  share of the sum 0.9247  information-not-capital
t-copula             nu 30.0  capital   6.2208  share of the sum 0.8887  information-not-capital
worst-case           nu None  capital   7.4467  share of the sum 1.0638  information-not-capital
simple sum 7: True | square-root row 6.0828: True | lower <= square-root <= worst: True
the docstring records the published example (Table 4, Pareto(2), d = 8, alpha 0.99): best VaR 9.00 to 9.00 | worst VaR 141.66 to 141.67


## State of the art

What the research found as the practice to go beyond (dossier 05, G.2 and G.3, with the status of each statement):

- The dossier **prescribes** the band, "instead of a single diversified number", and warns against claiming that diversification benefits are robust; this is a recommendation of the research, not a description of what supervisors or toolkits do.
- The Joint Forum found aggregation models "a work in progress" on which supervisors generally do not rely (the press release of 21 October 2010, https://www.bis.org/press/p101021.htm, gives both phrases).
- Rosenberg and Schuermann (2006) show, in one study, that the choice of copula matters modestly while the additive approximation overestimates risk by about 30 to 40 percent; and Embrechts, Puccetti and Rueschendorf (2013) give the sharp worst-case bounds, a method that exists in the literature.
- The research **found no maintained toolkit or published practice that ships the band beside every aggregate** (UNVERIFIED beyond the dossier's own search).

What exists, then, is the ingredients: the copulas, the bounds, the reference and the recommendation. What the case adds is to **compose them and never show one number alone**, as a rule enforced by contract (the case's requirements CT-1811 and CT-1812) and by the engine's return type.

In [6]:
# Read the requirements lines of the case that make the band a rule, to quote their identifiers and gates
requirements = (ROOT / "docs" / "design" / "features" / "c18" / "requirements.md").read_text(encoding="utf-8").splitlines()
for line in requirements:
    if line.startswith("| CT-1811") or line.startswith("| CT-1812"):
        identifier, text, gate = [cell.strip() for cell in line.strip("|").split("|")[:3]]
        print(identifier, "| the gate that fails when it is violated:", gate.split("::")[-1].strip("` "))

CT-1811 | the gate that fails when it is violated: test_the_headline_is_the_simple_sum_and_every_other_figure_is_information
CT-1812 | the gate that fails when it is violated: test_the_band_has_the_ladder_rows_and_the_euler_allocation_adds_up


## Beyond the state of the art

**BS-9, the dependence band shown with every aggregate.** The steps above are its evidence: the band is in every committed aggregate, in a fixed order, with one headline row and the rest labelled information; its closed-form rows are recomputed exactly; its structural rules pass; its engine returns the whole band; and the requirements CT-1811 and CT-1812 make it a gated rule of the case. The claim is stated as far as the evidence goes: **the case shows the band, always**. That the literature has no equivalent is UNVERIFIED beyond the research's search.

What the band adds over a single number is measurable on the committed document. The next cell reports, for the baseline aggregate, what each single figure would have said and what it would have hidden: the headline alone, the Gaussian figure alone, and the worst case alone, as distances to the rest of the band.

In [7]:
# What each single figure of the baseline aggregate would have said, and the share of the band it would have hidden
rows = {(r["method"], r["nu"]): r["capital_clp_m"] for r in blocks["baseline"]["band"]}
others = lambda key: [v for k, v in rows.items() if k != key]
for label, key in (("the headline alone (simple sum)", ("simple-sum", None)), ("the Gaussian figure alone", ("gaussian-copula", None)), ("the worst case alone", ("worst-case", None))):
    value = rows[key]
    below, above = sum(v < value for v in others(key)), sum(v > value for v in others(key))
    print(f"{label:34s} {value:10,.0f} | band rows below it: {below}, above it: {above} | range of the others as a multiple of it: {min(others(key)) / value:.2f} to {max(others(key)) / value:.2f}")
print("the band as shown: from", round(min(rows.values())), "to", round(max(rows.values())), "| the number of band rows above the headline:", sum(v > rows[("simple-sum", None)] for v in rows.values()))

the headline alone (simple sum)       658,046 | band rows below it: 6, above it: 1 | range of the others as a multiple of it: 0.18 to 1.25
the Gaussian figure alone             491,974 | band rows below it: 1, above it: 6 | range of the others as a multiple of it: 0.24 to 1.67
the worst case alone                  823,332 | band rows below it: 7, above it: 0 | range of the others as a multiple of it: 0.14 to 0.80
the band as shown: from 118518 to 823332 | the number of band rows above the headline: 1


## Results

The second part of this notebook is a statement about **what is not built**. The topic's cases are C18, built, and C19, the replication of the supervisory stress models against published loss rates ("known answer"), which **stays planned**. This notebook teaches nothing from C19. It reads the three records that say so: the case index lists C19 with a manifest path, the readiness record says that path is missing and gives the reason, and the engine's stress contract records why: no Federal Reserve portfolio type is replicable from the public record, so the engine has no supervisory replication function and no supervisory replication test. The next cell reads the three records and checks them against each other.

In [8]:
# Read the three records that say C19 is planned and not built, and check them against each other
index = read_json("manifests/index.json")
c19 = next(case for case in index["cases"] if case["case_id"] == "C19")
print("index:", c19["case_id"], "| kind:", c19["kind"], "| manifest path:", c19["manifest_path"], "| the manifest exists:", (DERIVED / c19["manifest_path"]).exists())
readiness = next(contract for contract in read_json("readiness.json")["contracts"] if contract["id"] == "case-manifest")
print("readiness record: state", readiness["state"], "| missing:", readiness["missing"], "| reason:", readiness["reason"]["en"])
contract_text = " ".join((ROOT / "docs" / "contracts" / "engine" / "stress-api.md").read_text(encoding="utf-8").split())
print("the stress contract records the decision to keep C19 planned:", "keeps C19 (supervisory stress model replication) planned" in contract_text)
print("it states that no supervisory replication function or test exists:", "no supervisory function and no supervisory replication test" in contract_text)
print("C18 built, C19 planned: the index lists", len([case for case in index["cases"] if case["category_id"] == "capital-stress"]), "cases in the capital-and-stress category,", "of which", sum((DERIVED / case["manifest_path"]).exists() for case in index["cases"] if case["category_id"] == "capital-stress"), "has a manifest")

index: C19 | kind: real | manifest path: manifests/C19.json | the manifest exists: False
readiness record: state partial | missing: ['manifests/C19.json'] | reason: 1 of 22 cases are planned and not built yet: C19.
the stress contract records the decision to keep C19 planned: True
it states that no supervisory replication function or test exists: True
C18 built, C19 planned: the index lists 2 cases in the capital-and-stress category, of which 1 has a manifest


- **The claim holds on every committed aggregate.** The five documents carry six aggregates (the correlation stress included), but **only two are distinct**: the aggregation block of the baseline, EBA, Fed and NGFS documents and of the t-copula document is one and the same block (the same hash), because the aggregation does not depend on the scenario, and the sixth is that block under the market-credit 0.8 stress. The evidence is therefore two aggregates, each with eight band rows in the stated order, exactly one row labelled headline capital (the simple sum, CLP 658,046 million in both) and the seven others labelled information, one horizon and one level (one year, 99.9 percent), an Euler gap between 0 and $1.5 \times 10^{-16}$, and a model-risk block labelled information. The worst case is 1.2512 times the headline in both. It does not follow that the band was checked under four stress scenarios: it was checked on the base aggregate and on the market-credit 0.8 stress.
- **The closed-form rows are recomputed exactly in all six blocks (two distinct aggregates)**, the bracket test and the sign test (on the diversified rows) are green in all six, and the stressed document's square-root row, 568,521, is above the base one, 528,106.
- **The engine returns the whole band on its golden example**: a `Band` of seven rows (two t degrees of freedom), headline 7.0, square-root row 6.0828 (the golden value), Gaussian 6.0912, t 6.4726 and 6.2208, lower bound 0.7601 and worst case 7.4467, which is 1.0638 times the simple sum. The docstring of `worst_var_bounds` records the published example (Table 4, Pareto(2) marginals, $d = 8$) with its worst-case range, which notebook 03 reproduced.
- **Each single figure hides most of the band.** The headline alone hides that the worst case is above it (6 rows below, 1 above) and that the others run from 0.18 to 1.25 of it; the Gaussian figure alone has 1 row below and 6 above and the others run from 0.24 to 1.67 of it; the worst case alone has all seven others below, down to 0.14 of it. The band spans a factor of seven, 118,518 to 823,332.
- **The replication of the supervisory stress models, C19, stays planned, and three records agree.** The index lists C19 with the path `manifests/C19.json`, which does not exist; the readiness record marks the case manifest `partial`, names that path as missing and gives the reason ("1 of 22 cases are planned and not built yet: C19"); the engine's stress contract records the decision to keep C19 planned and states that the engine has no supervisory replication function and no supervisory replication test. Of the two cases of the capital-and-stress category, one has a manifest.

What this notebook does not claim: that no one else shows a band (UNVERIFIED beyond the research's search); that the band is derived from first principles here (notebooks 02 and 03 rebuild the marginals from the committed quantile grids and get the committed rows back to within 0.01 percent; the million-draw samples behind the grids are not committed); or anything about C19's answer, which is not built.

## Summary

- BS-9 claims that every aggregate capital figure of C18 is shown with the whole dependence band, with one headline row and the rest labelled information; the committed documents carry it in two distinct aggregates (the base one, repeated in five documents, and the market-credit 0.8 stress) and every check of this notebook passes.
- The closed-form rows of each band are recomputed exactly by the engine; the bracket and sign rules hold; the engine's `dependence_band` returns a `Band`, never a scalar, on its own golden example.
- The claim is limited to what the case does; that nobody else ships the band is UNVERIFIED beyond the research's search.
- A single figure hides most of the band: the headline alone hides that the worst case is above it, and the band as a whole spans a factor of seven.
- The replication of the supervisory stress models, C19, stays planned: the three records agree, and nothing in this topic teaches from it.

**What to try next.** Add the sharp bounds to a band of your own marginals with `dependence_band` and look at the width of your own band before you look at a diversified figure.

## References

- Embrechts, P., Puccetti, G. and Rueschendorf, L. (2013), Model uncertainty and VaR aggregation, Journal of Banking and Finance 37(8). https://doi.org/10.1016/j.jbankfin.2013.03.014
- Rosenberg, J. V. and Schuermann, T. (2006), A general approach to integrated risk management with skewed, fat-tailed risks, Journal of Financial Economics 79(3). https://doi.org/10.1016/j.jfineco.2005.03.001
- Joint Forum (2010), Developments in modelling risk aggregation. https://www.bis.org/publ/joint25.pdf
- Bank for International Settlements (2010), press release of 21 October 2010 on the Joint Forum report Developments in modelling risk aggregation. https://www.bis.org/press/p101021.htm
- Dossier 05 of the persisted Contraste research (G.2 and G.3: the evidence on aggregation and the dependence pitfalls), as summarised in the SDD's research basis.
- This repository: `docs/design/features/c18/requirements.md` (CT-1811 and CT-1812), `docs/contracts/engine/aggregation-api.md` and `docs/contracts/engine/stress-api.md`.